<a href="https://colab.research.google.com/github/IvonneRomero1008/IA-Trabajos-escolares/blob/main/pp/Informe_resultados_encuesta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# INFORME Y GRÁFICAS DESDE EL CSV DE RESULTADOS


# 1. Instalar librerías
!pip install pandas matplotlib python-docx --quiet

# 2. Importar librerías
import pandas as pd
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Inches
from google.colab import files

# 3. Subir el archivo
print("Sube el archivo: Informe de encuesta.csv")
uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]

# 4. Leer el CSV
df = pd.read_csv(
    nombre_archivo,
    encoding="latin-1",
    sep=";",
    engine="python"
)

# 5. Limpiar columnas
df.columns = ["Pregunta", "Extra", "Respuesta", "Frecuencia", "Porcentaje"]

if "Extra" in df.columns:
    df = df.drop(columns=["Extra"])

df["Frecuencia"] = pd.to_numeric(df["Frecuencia"], errors="coerce")
df["Porcentaje"] = pd.to_numeric(
    df["Porcentaje"].astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

df = df.dropna(subset=["Frecuencia", "Porcentaje"])
df["Pregunta"] = df["Pregunta"].astype(str).str.strip()
df["Respuesta"] = df["Respuesta"].astype(str).str.strip()

print("Archivo cargado correctamente.")
print("Total de filas:", len(df))

# 6. Función para encontrar una pregunta por palabras clave
def buscar_pregunta(palabras_clave):
    for pregunta in df["Pregunta"].unique():
        texto = pregunta.lower()

        if all(palabra.lower() in texto for palabra in palabras_clave):
            return pregunta

    return None

# 7. Función para crear gráfica
def crear_grafica(palabras_clave, titulo, nombre_imagen):
    pregunta = buscar_pregunta(palabras_clave)

    if pregunta is None:
        print("No se encontró:", titulo)
        return None

    datos = df[df["Pregunta"] == pregunta].copy()
    datos = datos.sort_values("Frecuencia", ascending=True)

    plt.figure(figsize=(11, 5))
    plt.barh(
        datos["Respuesta"],
        datos["Frecuencia"],
        color="#4C72B0"
    )
    plt.title(titulo, fontsize=13)
    plt.xlabel("Número de respuestas")
    plt.xlim(0, datos["Frecuencia"].max() * 1.30)

    for i, (frecuencia, porcentaje) in enumerate(
        zip(datos["Frecuencia"], datos["Porcentaje"])
    ):
        plt.text(
            frecuencia + 0.15,
            i,
            f"{int(frecuencia)} ({porcentaje:.1f}%)",
            va="center"
        )

    plt.tight_layout()
    plt.savefig(nombre_imagen, dpi=200, bbox_inches="tight")
    plt.close()

    print("Gráfica creada:", nombre_imagen)
    return nombre_imagen

# 8. Crear las gráficas
imagenes = []

imagenes.append(
    (
        crear_grafica(
            ["habitualmente", "inteligencia artificial"],
            "Uso habitual de herramientas de IA generativa",
            "uso_ia.png"
        ),
        "Figura 1. Uso habitual de herramientas de IA generativa"
    )
)

imagenes.append(
    (
        crear_grafica(
            ["días a la semana", "herramientas de ia"],
            "Días por semana en que se utiliza IA",
            "dias_uso.png"
        ),
        "Figura 2. Días por semana en que se utiliza IA"
    )
)

imagenes.append(
    (
        crear_grafica(
            ["mensajes o instrucciones", "prompts"],
            "Número de prompts enviados por día",
            "prompts_dia.png"
        ),
        "Figura 3. Número de prompts enviados por día"
    )
)

imagenes.append(
    (
        crear_grafica(
            ["verificar la información", "antes de usarlos"],
            "Verificación de la información generada por IA",
            "verificacion.png"
        ),
        "Figura 4. Verificación de la información generada por IA"
    )
)

imagenes.append(
    (
        crear_grafica(
            ["entregas un trabajo", "sin realizar cambios"],
            "Entrega de trabajos con respuestas de IA sin cambios",
            "entrega_trabajos.png"
        ),
        "Figura 5. Entrega de trabajos con respuestas de IA sin cambios"
    )
)

imagenes.append(
    (
        crear_grafica(
            ["recibido previamente", "curso, taller o capacitación"],
            "Capacitación previa sobre el uso de IA",
            "capacitacion.png"
        ),
        "Figura 6. Capacitación previa sobre el uso de IA"
    )
)

imagenes.append(
    (
        crear_grafica(
            ["consultas repetitivas", "huella de carbono"],
            "Conocimiento del impacto ambiental de la IA",
            "impacto_ambiental.png"
        ),
        "Figura 7. Conocimiento del impacto ambiental de la IA"
    )
)

imagenes.append(
    (
        crear_grafica(
            ["optimizar tus prompts", "kwh"],
            "Disposición a optimizar prompts para reducir consumo",
            "disposicion_optimizar.png"
        ),
        "Figura 8. Disposición a optimizar prompts para reducir consumo"
    )
)

imagenes.append(
    (
        crear_grafica(
            ["tipo de talleres o cursos", "ofreciera la institución"],
            "Taller o curso más útil solicitado",
            "talleres_solicitados.png"
        ),
        "Figura 9. Taller o curso más útil solicitado"
    )
)

# 9. Crear documento Word
documento = Document()

documento.add_heading("Resultados de la encuesta", level=1)

documento.add_paragraph(
    "Se aplicó un cuestionario a 31 estudiantes con el propósito de conocer sus patrones "
    "de uso de herramientas de inteligencia artificial generativa, las dificultades que "
    "enfrentan al utilizarlas y sus necesidades de formación. La mayoría de los participantes "
    "manifestó utilizar habitualmente este tipo de herramientas: 28 personas (90.3%) respondió "
    "“sí”, mientras que 3 (9.7%) indicó no utilizarlas."
)

documento.add_paragraph(
    "Respecto a la frecuencia de uso, 18 participantes (58.1%) emplean IA entre uno y dos días "
    "por semana; 10 (32.3%) la utilizan de tres a cuatro días; y 3 (9.7%) la usan todos los días. "
    "En cuanto al número de consultas o prompts enviados en los días de uso, 18 personas (58.1%) "
    "reportan enviar menos de cinco; 7 (22.6%), entre cinco y quince; 3 (9.7%), entre dieciséis "
    "y treinta; y 3 (9.7%), más de treinta."
)

documento.add_heading("Principales usos académicos", level=2)

documento.add_paragraph(
    "Las respuestas muestran que la IA se utiliza principalmente para comprender temas académicos, "
    "programar, corregir código, redactar, resumir textos y preparar exámenes. En los ejemplos de "
    "prompts proporcionados, predominan las solicitudes relacionadas con la creación de guías de "
    "estudio, resúmenes de documentos, explicaciones paso a paso de ejercicios y simulacros de examen."
)

documento.add_paragraph(
    "Esto indica que los estudiantes no solo emplean la IA para obtener respuestas rápidas, sino "
    "también como apoyo para organizar información, repasar contenidos y comprender materias "
    "consideradas difíciles. No obstante, algunos prompts recogidos son poco específicos, como "
    "“Una guía” o “Apóyame a estudiar cierto tema”, lo cual puede reducir la calidad, precisión "
    "y utilidad de las respuestas obtenidas."
)

documento.add_heading("Verificación y errores", level=2)

documento.add_paragraph(
    "Sobre la verificación de la información generada por IA, 12 participantes (38.7%) señalaron "
    "que siempre validan fuentes, citas y datos; 8 (25.8%) indicaron que casi siempre revisan y "
    "adaptan la respuesta; 9 (29.0%) solo verifican cuando la respuesta parece sospechosa; y 2 "
    "(6.5%) manifestaron confiar directamente en la respuesta y copiarla sin modificarla."
)

documento.add_paragraph(
    "Asimismo, 15 estudiantes (48.4%) consideran que, de cada diez consultas, entre tres y cuatro "
    "no coinciden con lo esperado o contienen errores; 12 (38.7%) reportan entre cero y dos casos; "
    "y 4 (12.9%) entre cinco y seis. Estos resultados evidencian que, aunque una parte de los "
    "usuarios realiza algún tipo de revisión, todavía existe un riesgo importante de aceptar "
    "información incorrecta sin contrastarla con fuentes confiables."
)

documento.add_paragraph(
    "En relación con la entrega de trabajos, 16 personas (51.6%) admiten que a veces entregan una "
    "tarea con respuestas de IA sin realizar cambios significativos, 14 (45.2%) señalaron que nunca "
    "lo hacen y 1 (3.2%) indicó que lo hace con frecuencia. Este hallazgo refuerza la necesidad de "
    "promover el uso crítico de la IA, enfocado en revisar, adaptar y responsabilizarse por el "
    "contenido final."
)

documento.add_heading("Capacitación en IA", level=2)

documento.add_paragraph(
    "Un resultado relevante es que 21 participantes (67.7%) no han recibido ningún curso, taller o "
    "capacitación sobre el uso correcto de la IA; 8 (25.8%) se han capacitado por cuenta propia y "
    "solo 2 (6.5%) han recibido una capacitación impartida por la universidad."
)

documento.add_paragraph(
    "Esta falta de formación institucional coincide con las dificultades reportadas para identificar "
    "errores, formular instrucciones precisas y verificar la información generada. Por ello, los "
    "resultados sustentan la necesidad de incorporar talleres o cursos que desarrollen competencias "
    "de alfabetización en IA entre los estudiantes."
)

documento.add_heading("Impacto ambiental y disposición a optimizar", level=2)

documento.add_paragraph(
    "Respecto al consumo energético y ambiental de la IA, 21 participantes (67.7%) afirmaron conocer "
    "el impacto energético y ambiental de estas herramientas; 5 (16.1%) lo intuían sin conocer su "
    "magnitud; y 5 (16.1%) pensaban que las consultas digitales no consumen recursos significativos."
)

documento.add_paragraph(
    "Al preguntarles por su disposición a optimizar sus prompts si contaran con una herramienta que "
    "mostrara el consumo estimado en kWh y CO2, 23 personas (74.2%) seleccionaron el nivel máximo "
    "de disposición, 5 (16.1%) eligieron el nivel 4, y las respuestas restantes se ubicaron en "
    "niveles menores. En conjunto, 28 de 31 participantes (90.3%) mostraron una disposición alta, "
    "correspondiente a los niveles 4 o 5, para modificar sus consultas y reducir el consumo energético."
)

documento.add_heading("Necesidades de formación", level=2)

documento.add_paragraph(
    "Cuando se les preguntó qué tipo de taller o curso sería más útil, 16 participantes (51.6%) "
    "eligieron ingeniería de prompts, es decir, aprender a redactar instrucciones precisas para "
    "obtener mejores respuestas desde la primera consulta. En segundo lugar, 8 personas (25.8%) "
    "solicitaron contenidos sobre detección de errores, verificación de fuentes y alucinaciones."
)

# Tabla
tabla = documento.add_table(rows=1, cols=3)
tabla.style = "Light Grid Accent 1"

encabezados = ["Tipo de taller o curso solicitado", "Frecuencia", "Porcentaje"]

for i, texto in enumerate(encabezados):
    tabla.rows[0].cells[i].text = texto

datos_tabla = [
    ["Ingeniería de prompts", "16", "51.6%"],
    ["Detección de errores y verificación de fuentes", "8", "25.8%"],
    ["Costo energético, ambiental y financiero de la IA", "4", "12.9%"],
    ["IA aplicada a áreas específicas", "3", "9.7%"],
]

for fila in datos_tabla:
    celdas = tabla.add_row().cells
    for i, valor in enumerate(fila):
        celdas[i].text = valor

documento.add_heading("Conclusión del análisis", level=2)

documento.add_paragraph(
    "Los resultados indican que la IA generativa forma parte de las actividades académicas de la "
    "mayoría de los estudiantes encuestados, especialmente para estudiar, resolver dudas, elaborar "
    "resúmenes y preparar exámenes. Sin embargo, su uso todavía presenta debilidades relacionadas "
    "con la verificación de información, la reformulación de consultas y la revisión de las "
    "respuestas antes de utilizarlas en trabajos escolares."
)

documento.add_paragraph(
    "La baja proporción de estudiantes con capacitación formal, de solo 6.5%, y la alta demanda de "
    "formación en ingeniería de prompts y detección de errores justifican el desarrollo de una "
    "propuesta de capacitación institucional. Esta debería enfocarse en el uso responsable de la IA, "
    "la redacción de instrucciones efectivas, la verificación de fuentes y la consideración del "
    "impacto ambiental asociado al uso repetido de estas herramientas."
)

# 10. Insertar solamente las gráficas que sí se generaron
for imagen, titulo in imagenes:
    if imagen is not None:
        documento.add_picture(imagen, width=Inches(6))
        parrafo = documento.add_paragraph(titulo)
        parrafo.alignment = 1

# 11. Guardar y descargar
nombre_documento = "Informe_resultados_encuesta_IA.docx"
documento.save(nombre_documento)

files.download(nombre_documento)

print("\n¡Listo! Se descargó:", nombre_documento)

Sube el archivo: Informe de encuesta.csv


Saving Informe de encuesta.csv to Informe de encuesta (4).csv
Archivo cargado correctamente.
Total de filas: 132
Gráfica creada: uso_ia.png
Gráfica creada: dias_uso.png
Gráfica creada: prompts_dia.png
Gráfica creada: verificacion.png
Gráfica creada: entrega_trabajos.png
Gráfica creada: capacitacion.png
Gráfica creada: impacto_ambiental.png
Gráfica creada: disposicion_optimizar.png
Gráfica creada: talleres_solicitados.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


¡Listo! Se descargó: Informe_resultados_encuesta_IA.docx
